In [1]:
from fastembed import SparseTextEmbedding
import pandas as pd

/mnt/D/Data_Science/Common/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

text = "OpenSearch and Qdrant are used for hybrid retrieval in RAG systems."
query = "hybrid retrieval Qdrant RAG"


In [3]:

model = SparseTextEmbedding(model_name="Qdrant/bm25")


Fetching 18 files: 100%|██████████| 18/18 [00:01<00:00, 15.59it/s]


In [4]:

doc_emb = list(model.embed([text]))[0]
query_emb = list(model.embed([query]))[0]


In [5]:

def sparse_to_df(emb, side):
    return pd.DataFrame({
        "token_id": list(emb.indices),
        "weight": [float(v) for v in emb.values],
        "side": side
    })


In [6]:

doc_df = sparse_to_df(doc_emb, "doc")
query_df = sparse_to_df(query_emb, "query")


In [7]:

merged = doc_df.merge(query_df, on="token_id", how="outer", suffixes=("_doc", "_query")).fillna(0)
merged["overlap"] = (merged["weight_doc"] > 0) & (merged["weight_query"] > 0)
merged["score_product"] = merged["weight_doc"] * merged["weight_query"]
merged = merged.sort_values(
    by=["overlap", "score_product", "weight_doc", "weight_query"],
    ascending=False
)


In [8]:

print("DOCUMENT")
print(text)
print("\nQUERY")
print(query)


DOCUMENT
OpenSearch and Qdrant are used for hybrid retrieval in RAG systems.

QUERY
hybrid retrieval Qdrant RAG


In [9]:

print("\nDOC non-zero terms:", len(doc_emb.indices))
print("QUERY non-zero terms:", len(query_emb.indices))



DOC non-zero terms: 7
QUERY non-zero terms: 4


In [10]:

print("\nTop matching token ids:")
print(merged.head(20).to_string(index=False))


Top matching token ids:
  token_id  weight_doc side_doc  weight_query side_query  overlap  score_product
 802574768    1.660867      doc      1.674197      query     True       2.780619
1041526299    1.660867      doc      1.674197      query     True       2.780619
1644075694    1.660867      doc      1.674197      query     True       2.780619
1839534477    1.660867      doc      1.674197      query     True       2.780619
 134102889    1.660867      doc      0.000000          0    False       0.000000
 640124220    1.660867      doc      0.000000          0    False       0.000000
2095749492    1.660867      doc      0.000000          0    False       0.000000
